# Chapter 2 Lab — Sleep Architecture & AASM Scoring

**Think of it like this:** sleep is an elevator ride down through a building, then back up.
- **Lobby (Wake)** — the crowd's awake and chatting.
- **Floor N1** — the doors just closed. Still a little aware.
- **Floor N2** — moving down steadily. Every so often there's a quick buzzy burst (a **spindle**) and one sharp spike (a **K-complex**) — like little "ding" sounds as the elevator passes markers.
- **Basement N3** — the deepest floor. Big, slow, rolling waves, like a deep engine rumble. Hardest floor to wake someone from.
- **The Dream Wing (REM)** — not really a floor at all, more a separate wing. The brain looks almost as busy as Wake, but the body is locked so you can't act out the dream.

This lab: load real 30-second windows **with the expert's actual answer attached**, guess the floor yourself first, then check.

If you haven't run Chapter 1's lab in this session, run the setup cell below first — it's the same download step.

In [ ]:
!pip install -q mne
%matplotlib inline
import mne
from mne.datasets.sleep_physionet.age import fetch_data

ALICE = 0
alice_files = fetch_data(subjects=[ALICE], recording=[1])[0]
raw = mne.io.read_raw_edf(
    alice_files[0], stim_channel="Event marker", infer_types=True, preload=True, verbose="error",
)
annot = mne.read_annotations(alice_files[1])
raw.set_annotations(annot, emit_warning=False)

## Turn the night into labelled 30-second epochs

AASM scoring works in fixed 30-second windows. Below, `annotation_desc_2_event_id` is just a lookup table from the expert's text labels ("Sleep stage W", "Sleep stage 1"...) to numbers MNE can work with. We crop the very long pre-sleep and post-sleep wake sections down to 30 minutes each, just so the dataset isn't mostly "awake and bored."

In [ ]:
annotation_desc_2_event_id = {
    "Sleep stage W": 1,
    "Sleep stage 1": 2,
    "Sleep stage 2": 3,
    "Sleep stage 3": 4,
    "Sleep stage 4": 4,  # old scoring rules split deep sleep into 3+4; modern AASM merges them into N3
    "Sleep stage R": 5,
}

annot.crop(annot[1]["onset"] - 30 * 60, annot[-2]["onset"] + 30 * 60)
raw.set_annotations(annot, emit_warning=False)

events, _ = mne.events_from_annotations(
    raw, event_id=annotation_desc_2_event_id, chunk_duration=30.0
)

event_id = {
    "Wake": 1, "N1": 2, "N2": 3, "N3": 4, "REM": 5,
}
id_to_name = {v: k for k, v in event_id.items()}

tmax = 30.0 - 1.0 / raw.info["sfreq"]
epochs = mne.Epochs(raw=raw, events=events, event_id=event_id, tmin=0.0, tmax=tmax, baseline=None, preload=True)
print(epochs)

## Guess before you peek

Run the next cell. It shows one random 30-second epoch **without telling you the label**. Look at the shape — fast and small? Big slow rolling waves? A sharp spike? — and write down your guess before running the cell after it, which reveals the real answer.

Re-run both cells a bunch of times (different random epochs each time) until you're getting most of them right.

In [ ]:
import random
idx = random.randrange(len(epochs))
epochs[idx].plot(scalings="auto", title="Guess the stage...");

In [ ]:
true_label = id_to_name[epochs.events[idx, 2]]
print("The real answer was:", true_label)

## A note on N1 vs. REM

If you keep mixing these two up, you're in good company — **N1 and REM are the hardest pair for human sleep technicians too**, and it stays the hardest pair for every classical and deep-learning model in this whole course. Don't treat your own confusion here as a sign you're doing something wrong.

## Done when...

You can run the guess-then-reveal cells above on a fresh handful of epochs and get most of them right, **for the same reasons** a human scorer would (the shape, not luck). That's the actual skill Chapter 4's classifier is about to try to automate.